# Stage 2 — Anomaly Classifier v5
### Fixes in v5
- **Bug 1 fixed**: Edge teeth whose mask slice is smaller than the crop box were previously skipped (`shape mismatch → continue`). Now the mask is zero-padded to match the crop size so ALL teeth get colored.
- **Bug 2 fixed**: `number_teeth_in_quadrants` previously capped each quadrant at 8 and silently dropped the rest. Now ALL detected teeth are numbered (9, 10… if Stage 1 finds extras) and appear in the report.

In [ ]:
# ================== CELL 1: PATHS & IMPORTS ==================
import os, cv2, torch, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from torchvision.ops import nms
import torchvision.transforms.functional as TF

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Dataset paths ──────────────────────────────────────────────
STAGE1_WEIGHTS = '/kaggle/input/maskrcnn-teeth-stage1-weights/maskrcnn_teeth_best.pth'
DENTEX_DIR     = '/kaggle/input/dentex-challenge-2023'

# Auto-detect correct subfolder name (dashes vs underscores vary by dataset version)
import glob as _g
_candidates = _g.glob(os.path.join(DENTEX_DIR, '**'), recursive=True)
print('\nDENTEX subfolders found:')
for _c in sorted(_candidates):
    if os.path.isdir(_c): print(' ', _c)

# ── Set these after checking the printout above ─────────────────
DENTEX_TRAIN_IMG   = os.path.join(DENTEX_DIR, 'quadrant-enumeration-disease', 'xrays')
DENTEX_LABEL_DIR   = os.path.join(DENTEX_DIR, 'quadrant-enumeration-disease', 'train', 'labels')
INFER_IMG_DIR      = os.path.join(DENTEX_DIR, 'validation_data', 'validation_data',
                                   'quadrant_enumeration_disease', 'xrays')

ANOMALY_CLASSES = {
    0: 'background',
    1: 'Caries',
    2: 'Deep Caries',
    3: 'Periapical Lesion',
    4: 'Impacted',
}
NUM_ANOMALY_CLASSES = len(ANOMALY_CLASSES)  # 5 (incl. background)
print(f'\nAnomaly classes: {ANOMALY_CLASSES}')
print(f'Train images : {DENTEX_TRAIN_IMG}')
print(f'Inference dir: {INFER_IMG_DIR}')

In [ ]:
# ================== CELL 2: STAGE 1 MODEL (Mask R-CNN) ==================
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

def get_stage1_model(num_classes=33):
    model = maskrcnn_resnet50_fpn(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    in_feat_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_feat_mask, 256, num_classes)
    return model

stage1 = get_stage1_model(num_classes=33)
stage1.load_state_dict(torch.load(STAGE1_WEIGHTS, map_location=DEVICE))
stage1.to(DEVICE).eval()
print('[CELL 2] Stage 1 Mask R-CNN loaded.')

In [ ]:
# ================== CELL 3: DENTEX DATASET ==================
import json
from torch.utils.data import Dataset, DataLoader

class DentexDataset(Dataset):
    def __init__(self, img_dir, ann_file, transforms=None):
        self.img_dir    = img_dir
        self.transforms = transforms
        with open(ann_file) as f:
            data = json.load(f)
        self.imgs  = {i['id']: i for i in data['images']}
        self.anns  = {}
        for a in data.get('annotations', []):
            self.anns.setdefault(a['image_id'], []).append(a)
        self.ids = list(self.imgs.keys())

    def __len__(self): return len(self.ids)

    def __getitem__(self, idx):
        img_id  = self.ids[idx]
        img_inf = self.imgs[img_id]
        img_pil = Image.open(os.path.join(self.img_dir,
                             img_inf['file_name'])).convert('RGB')
        img_t   = TF.to_tensor(img_pil)
        anns    = self.anns.get(img_id, [])
        boxes, labels = [], []
        for a in anns:
            x, y, w, h = a['bbox']
            boxes.append([x, y, x+w, y+h])
            labels.append(a['category_id'])
        target = {
            'boxes':  torch.tensor(boxes,  dtype=torch.float32) if boxes
                      else torch.zeros((0,4), dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.int64)   if labels
                      else torch.zeros((0,),  dtype=torch.int64),
        }
        if self.transforms:
            img_t = self.transforms(img_t)
        return img_t, target

print('[CELL 3] DentexDataset defined.')

In [ ]:
# ================== CELL 4: STAGE 2 MODEL (Faster R-CNN) ==================
from torchvision.models.detection import fasterrcnn_resnet50_fpn

def get_stage2_model(num_classes=5):
    model = fasterrcnn_resnet50_fpn(weights='DEFAULT')
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

stage2 = get_stage2_model(num_classes=NUM_ANOMALY_CLASSES)
stage2.to(DEVICE)
print('[CELL 4] Stage 2 Faster R-CNN initialised.')

In [ ]:
# ================== CELL 5: TRAIN STAGE 2 ==================
from torch.optim import SGD
from torch.optim.lr_scheduler import StepLR

ANN_FILE   = os.path.join(DENTEX_DIR, 'quadrant-enumeration-disease',
                          'train', 'annotations.json')
TRAIN_IMGS = os.path.join(DENTEX_DIR, 'quadrant-enumeration-disease', 'xrays')

dataset    = DentexDataset(TRAIN_IMGS, ANN_FILE)
loader     = DataLoader(dataset, batch_size=2, shuffle=True,
                        collate_fn=lambda b: tuple(zip(*b)))

optimizer  = SGD(stage2.parameters(), lr=0.005, momentum=0.9, weight_decay=1e-4)
scheduler  = StepLR(optimizer, step_size=10, gamma=0.1)
NUM_EPOCHS = 30

stage2.train()
for epoch in range(NUM_EPOCHS):
    total_loss = 0.0
    for imgs, targets in loader:
        imgs    = [i.to(DEVICE) for i in imgs]
        targets = [{k: v.to(DEVICE) for k,v in t.items()} for t in targets]
        loss_d  = stage2(imgs, targets)
        loss    = sum(loss_d.values())
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    if (epoch+1) % 5 == 0:
        print(f'  Epoch [{epoch+1}/{NUM_EPOCHS}]  loss={total_loss/len(loader):.4f}')

torch.save(stage2.state_dict(), '/kaggle/working/stage2_anomaly_best.pth')
print('[CELL 5] Stage 2 training complete. Model saved.')

In [ ]:
# ================== CELL 6: STAGE 2 INFERENCE HELPER ==================
def infer_stage2(model, img_tensor, conf=0.4):
    model.eval()
    with torch.no_grad():
        pred = model([img_tensor.to(DEVICE)])[0]
    keep = pred['scores'] >= conf
    return {k: v[keep].cpu().numpy() for k in ['boxes','labels','scores']}

print('[CELL 6] Stage 2 inference helper defined.')

In [ ]:
# ================== CELL 7: PER-TOOTH PIPELINE (v5 — both bugs fixed) ==================
# FIX 1: color_teeth — pad mask with zeros instead of skipping edge teeth
# FIX 2: number_teeth_in_quadrants — removed cap-at-8 drop, number ALL teeth

VIBRANT = [
    [255,  0,   0], [0,   255,   0], [0,    0,  255], [255, 255,   0],
    [255,  0, 255], [0,   255, 255], [255, 128,   0], [128,   0, 255],
    [255,  0, 128], [0,   255, 128], [128, 255,   0], [0,   128, 255],
]

def run_inference_s1(img_tensor, conf=0.5, iou_thr=0.3):
    with torch.no_grad():
        pred = stage1([img_tensor.to(DEVICE)])[0]
    keep = pred['scores'] >= conf
    pred = {k: v[keep] for k, v in pred.items()}
    if len(pred['boxes']):
        idx  = nms(pred['boxes'], pred['scores'], iou_thr)
        pred = {k: v[idx] for k, v in pred.items()}
    return {k: pred[k].cpu().numpy() for k in ['boxes','labels','masks','scores']}

def crop_to_teeth_region(image_np, predictions, padding=20):
    boxes = predictions['boxes']
    if len(boxes) == 0:
        return image_np, None
    x1 = max(0, int(boxes[:,0].min()) - padding)
    y1 = max(0, int(boxes[:,1].min()) - padding)
    x2 = min(image_np.shape[1], int(boxes[:,2].max()) + padding)
    y2 = min(image_np.shape[0], int(boxes[:,3].max()) + padding)
    return image_np[y1:y2, x1:x2], (x1, y1, x2, y2)

def color_teeth(cropped_np, predictions, crop_box):
    colored = cropped_np.copy()
    if crop_box is None:
        return colored
    x1, y1, x2, y2 = crop_box
    crop_h = colored.shape[0]
    crop_w = colored.shape[1]
    for i, mask in enumerate(predictions['masks']):
        mask_binary  = (mask[0] > 0.5).astype(np.uint8)
        mask_cropped = mask_binary[y1:y2, x1:x2]
        if mask_cropped.shape[0] == 0 or mask_cropped.shape[1] == 0:
            continue
        # FIX 1: pad instead of skip when mask slice < crop size (edge teeth)
        if mask_cropped.shape[0] < crop_h or mask_cropped.shape[1] < crop_w:
            padded = np.zeros((crop_h, crop_w), dtype=np.uint8)
            ph = min(mask_cropped.shape[0], crop_h)
            pw = min(mask_cropped.shape[1], crop_w)
            padded[:ph, :pw] = mask_cropped[:ph, :pw]
            mask_cropped = padded
        if mask_cropped.shape[:2] != colored.shape[:2]:
            continue  # genuine full mismatch — skip
        color   = VIBRANT[i % len(VIBRANT)]
        overlay = colored.copy()
        for c in range(3):
            overlay[:,:,c] = np.where(mask_cropped == 1, color[c], overlay[:,:,c])
        colored = cv2.addWeighted(colored, 0.3, overlay, 0.7, 0)
        cnts, _ = cv2.findContours(mask_cropped, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(colored, cnts, -1, color, 3)
    return colored

def split_into_quadrants(predictions, image_shape, crop_box=None):
    if crop_box:
        x1, y1, x2, y2 = crop_box
        center_x = (x1 + x2) // 2
        center_y = (y1 + y2) // 2
    else:
        center_x = image_shape[1] // 2
        center_y = image_shape[0] // 2
    quadrants = {'UL':[], 'UR':[], 'LL':[], 'LR':[]}
    for i, (box, lbl, mask, sc) in enumerate(zip(
            predictions['boxes'], predictions['labels'],
            predictions['masks'], predictions['scores'])):
        cx = int((box[0] + box[2]) / 2)
        cy = int((box[1] + box[3]) / 2)
        if   cx <  center_x and cy <  center_y: q = 'UL'
        elif cx >= center_x and cy <  center_y: q = 'UR'
        elif cx <  center_x and cy >= center_y: q = 'LL'
        else:                                    q = 'LR'
        quadrants[q].append({
            'index': i, 'label': int(lbl), 'box': box,
            'centroid': (cx, cy), 'mask': mask, 'score': float(sc)
        })
    return quadrants, (center_x, center_y)

def number_teeth_in_quadrants(quadrants):
    numbered = {}
    for q_name, teeth in quadrants.items():
        if not teeth:
            numbered[q_name] = []
            continue
        # Sort from midline outward
        if q_name in ('UL', 'LL'):
            sorted_teeth = sorted(teeth, key=lambda t: -t['centroid'][0])
        else:
            sorted_teeth = sorted(teeth, key=lambda t:  t['centroid'][0])
        # FIX 2: number ALL teeth — removed cap-at-8 drop
        for num, tooth in enumerate(sorted_teeth, 1):
            tooth['number']   = num
            tooth['quadrant'] = q_name
        numbered[q_name] = sorted_teeth
    return numbered

def detect_anomalies(img_pil, numbered_quads, img_w, img_h):
    anomaly_map = {}
    for q_name, teeth in numbered_quads.items():
        for tooth in teeth:
            bx1, by1, bx2, by2 = [int(v) for v in tooth['box']]
            tx1 = max(0, bx1-8);       ty1 = max(0, by1-8)
            tx2 = min(img_w, bx2+8);   ty2 = min(img_h, by2+8)
            crop = img_pil.crop((tx1, ty1, tx2, ty2))
            if crop.width < 4 or crop.height < 4:
                continue
            s2 = infer_stage2(stage2, TF.to_tensor(crop))
            if len(s2['boxes']) > 0:
                tooth_anoms = [
                    {
                        'label':      int(lb),
                        'label_name': ANOMALY_CLASSES.get(int(lb), f'cls_{lb}'),
                        'score':      float(sc)
                    }
                    for lb, sc in zip(s2['labels'], s2['scores'])
                    if ANOMALY_CLASSES.get(int(lb), '') != 'background'
                ]
                if tooth_anoms:
                    anomaly_map[(q_name, tooth['number'])] = {
                        'anomalies': tooth_anoms,
                        'box':       tooth['box']
                    }
    return anomaly_map

def run_full_pipeline(img_path):
    img_pil = Image.open(img_path).convert('RGB')
    img_np  = np.array(img_pil)
    H, W    = img_np.shape[:2]
    img_t   = TF.to_tensor(img_pil)
    predictions = run_inference_s1(img_t)
    if len(predictions['boxes']) == 0:
        print('  WARNING: Stage 1 found no teeth.'); return None
    cropped_np, crop_box   = crop_to_teeth_region(img_np, predictions)
    colored_np             = color_teeth(cropped_np, predictions, crop_box)
    quadrants, center      = split_into_quadrants(predictions, img_np.shape, crop_box)
    numbered_quads         = number_teeth_in_quadrants(quadrants)
    anomaly_map            = detect_anomalies(img_pil, numbered_quads, W, H)
    total = sum(len(v) for v in numbered_quads.values())
    print(f'  Stage 1: {len(predictions["boxes"])} detected | Numbered: {total} | Anomalies: {len(anomaly_map)}')
    return img_np, cropped_np, colored_np, numbered_quads, anomaly_map, center, crop_box

print('[CELL 7] v5 pipeline defined — edge teeth fixed, all teeth numbered.')

In [ ]:
# ================== CELL 8: VISUALIZATION ==================
import matplotlib.patches as mpatches

ANOMALY_COLORS = {
    'Caries':            (255,  50,  50),
    'Periapical Lesion': (255, 200,   0),
    'Deep Caries':       (200,   0, 200),
    'Impacted':          (255, 140,   0),
    'Healthy':           (  0, 220, 100),
}
BOX_HEALTHY_COLOR = (0, 220, 100)
BOX_ANOMALY_WIDTH = 5
BOX_HEALTHY_WIDTH = 1

def _put_label_pill(img, text, x1, y1, color):
    font = cv2.FONT_HERSHEY_SIMPLEX
    fs   = 0.65; thick = 2
    (tw, th), _ = cv2.getTextSize(text, font, fs, thick)
    pad  = 5
    rx1, ry1 = x1, max(0, y1 - th - 2*pad)
    rx2, ry2 = x1 + tw + 2*pad, y1
    cv2.rectangle(img, (rx1, ry1), (rx2, ry2), color, -1)
    lum     = 0.299*color[0] + 0.587*color[1] + 0.114*color[2]
    txt_clr = (0,0,0) if lum > 140 else (255,255,255)
    cv2.putText(img, text, (rx1+pad, ry2-pad), font, fs, txt_clr, thick)

def visualize_pipeline(img_path):
    result = run_full_pipeline(img_path)
    if result is None:
        return
    img_np, cropped_np, colored_np, numbered_quads, anomaly_map, center, crop_box = result
    result_img = colored_np.copy()
    h, w       = result_img.shape[:2]
    ox         = crop_box[0] if crop_box else 0
    oy         = crop_box[1] if crop_box else 0
    # Quadrant dividers
    cx_crop = center[0] - ox
    cy_crop = center[1] - oy
    cv2.line(result_img, (cx_crop, 0),  (cx_crop, h), (255, 255, 0), 5)
    cv2.line(result_img, (0, cy_crop),  (w, cy_crop), (255, 255, 0), 5)
    # Draw boxes and tooth numbers
    for q_name, teeth in numbered_quads.items():
        for tooth in teeth:
            cx  = tooth['centroid'][0] - ox
            cy  = tooth['centroid'][1] - oy
            bx1 = max(0, int(tooth['box'][0]) - ox)
            by1 = max(0, int(tooth['box'][1]) - oy)
            bx2 = min(w, int(tooth['box'][2]) - ox)
            by2 = min(h, int(tooth['box'][3]) - oy)
            key        = (q_name, tooth['number'])
            is_anomaly = key in anomaly_map
            if is_anomaly:
                anom      = anomaly_map[key]['anomalies'][0]
                box_color = ANOMALY_COLORS.get(anom['label_name'], (255, 50, 50))
                cv2.rectangle(result_img, (bx1, by1), (bx2, by2), box_color, BOX_ANOMALY_WIDTH)
                _put_label_pill(result_img,
                                f"{anom['label_name']} {anom['score']*100:.0f}%",
                                bx1, by1, box_color)
                num_color = box_color
            else:
                cv2.rectangle(result_img, (bx1, by1), (bx2, by2), BOX_HEALTHY_COLOR, BOX_HEALTHY_WIDTH)
                num_color = (0, 255, 255)
            if 0 <= cx < w and 0 <= cy < h:
                num = str(tooth['number'])
                cv2.putText(result_img, num, (cx-25, cy+20), cv2.FONT_HERSHEY_SIMPLEX, 2.0, (0,0,0),     8)
                cv2.putText(result_img, num, (cx-25, cy+20), cv2.FONT_HERSHEY_SIMPLEX, 2.0, (255,255,255), 5)
                cv2.putText(result_img, num, (cx-25, cy+20), cv2.FONT_HERSHEY_SIMPLEX, 2.0, num_color,    3)
    # Quadrant corner labels
    for label, pos in [('UL',(40,60)), ('UR',(w-120,60)), ('LL',(40,h-40)), ('LR',(w-120,h-40))]:
        cv2.putText(result_img, label, pos, cv2.FONT_HERSHEY_SIMPLEX, 2.5, (0,0,0),     8)
        cv2.putText(result_img, label, pos, cv2.FONT_HERSHEY_SIMPLEX, 2.5, (0,255,255), 5)
    # 4-panel figure
    fig = plt.figure(figsize=(24, 17))
    fig.patch.set_facecolor('#0e0e0e')
    gs  = fig.add_gridspec(2, 2, hspace=0.15, wspace=0.15)
    ax1 = fig.add_subplot(gs[0,0])
    ax1.imshow(img_np, cmap='gray')
    ax1.set_title('(1) Original Panoramic X-ray', color='white', fontsize=16, fontweight='bold'); ax1.axis('off')
    ax2 = fig.add_subplot(gs[0,1])
    ax2.imshow(cropped_np, cmap='gray')
    ax2.set_title('(2) Cropped to Teeth Region', color='white', fontsize=16, fontweight='bold'); ax2.axis('off')
    ax3 = fig.add_subplot(gs[1,0])
    ax3.imshow(colored_np)
    ax3.set_title('(3) Segmented & Colored Teeth  [each color = one tooth]', color='white', fontsize=16, fontweight='bold'); ax3.axis('off')
    ax4 = fig.add_subplot(gs[1,1])
    ax4.imshow(result_img)
    ax4.set_title('(4) Quadrants  |  Tooth Numbers  |  Anomaly Detection\n'
                  '    cyan number = Healthy   |   colored thick box = Anomaly',
                  color='white', fontsize=14, fontweight='bold'); ax4.axis('off')
    # Legend
    legend_elements = []
    for cls_name, rgb in ANOMALY_COLORS.items():
        if cls_name == 'Healthy': continue
        legend_elements.append(mpatches.FancyBboxPatch(
            (0,0), 1, 1, boxstyle='square,pad=0.1',
            facecolor='none', edgecolor=tuple(c/255 for c in rgb),
            linewidth=4, label=f'[ ] {cls_name} (anomaly box)'
        ))
    legend_elements.append(mpatches.FancyBboxPatch(
        (0,0), 1, 1, boxstyle='square,pad=0.1',
        facecolor='none', edgecolor=(0, 220/255, 100/255),
        linewidth=1.5, label='[ ] Healthy tooth (thin box)'
    ))
    legend = fig.legend(
        handles=legend_elements, loc='lower center',
        ncol=len(legend_elements), fontsize=12,
        facecolor='#1a1a1a', labelcolor='white',
        framealpha=0.9, edgecolor='#444',
        title='--- Anomaly Bounding Box Colors  (Panel 4 only) ---',
        title_fontsize=12,
    )
    legend.get_title().set_color('#aaaaaa')
    fname = '/kaggle/working/pipeline_result_' + os.path.basename(img_path).replace(' ','_') + '.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor='#0e0e0e')
    plt.show()
    print(f'  Saved -> {fname}')
    # Clinical report
    print('\n' + '='*70)
    print('  CLINICAL ANOMALY REPORT  --  ' + os.path.basename(img_path))
    print('='*70)
    all_teeth = [t for teeth in numbered_quads.values() for t in teeth]
    for tooth in sorted(all_teeth, key=lambda t: (t['quadrant'], t['number'])):
        key = (tooth['quadrant'], tooth['number'])
        if key not in anomaly_map:
            print(f"  Tooth {tooth['number']:>2}  [{tooth['quadrant']}]  ->  Healthy")
        else:
            for a in anomaly_map[key]['anomalies']:
                print(f"  Tooth {tooth['number']:>2}  [{tooth['quadrant']}]  ->  !! {a['label_name']:<22} [{a['score']*100:.0f}%]")
    print('='*70)
    print(f'  Summary: {len(anomaly_map)} anomalous / {len(all_teeth)} total teeth')
    print('='*70)

print('[CELL 8] Visualization defined.')

In [ ]:
# ================== CELL 9: RUN INFERENCE ==================
import glob

xray_files = sorted(
    glob.glob(os.path.join(INFER_IMG_DIR, '*.png')) +
    glob.glob(os.path.join(INFER_IMG_DIR, '*.jpg'))
)
print(f'[CELL 9] Found {len(xray_files)} panoramic X-rays')

for i, fpath in enumerate(xray_files[:3]):
    print(f'\n{"="*60}\nSAMPLE {i+1}: {os.path.basename(fpath)}\n{"="*60}')
    visualize_pipeline(fpath)